# Loading data

We use a wrapper on top of HuggingFace Datasets to load the AssayBench datasets.

Here we show how to load the temporal train/val/test split, as well as the LaTest split.

In [1]:
from assaybench.dataset.dataset import AssayBenchDataset

In [2]:
dataset_name = "biogrid"
novel_dataset_name = "LaTest"  # set to None if not using a novel dataset
split_type = "year"  # or "random"
fold = 0  # which fold to use in the given split type

ds = AssayBenchDataset(
                dataset_name=dataset_name,
                novel_dataset_name=novel_dataset_name,
                split_type=split_type,
                fold=fold,
            )


train,val,test, novel = ds.get_train_test_split()
print(f"Number of screens in train: {len(train)}")
print(f"Number of screens in val: {len(val)}")
print(f"Number of screens in test: {len(test)}")
print(f"Number of screens in novel dataset: {len(novel)}")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Genentech/assaybench/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Genentech/assaybench/988daee0619a40386bca661a31b3859fb01ee1c4/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/Genentech/assaybench/988daee0619a40386bca661a31b3859fb01ee1c4/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Genentech/assaybench/resolve/988daee0619a40386bca661a31b3859fb01ee1c4/assaybench.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Genentech/assaybench/Genentech/assaybench.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/Genentech/assaybench/revision/988daee0619a40386bca661a31b3859fb01ee1c4 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Genentech/assaybench/resolve/988daee0619a40386bca661a31b3859fb01ee1c4/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=Genentech/assaybench "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/Genentech/assaybench/tree/988daee0619a40386bca661a31b3859fb01ee1c4/LaTest?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET

Generating train split:   0%|          | 0/1901 [00:00<?, ? examples/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Genentech/assaybench/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Genentech/assaybench/988daee0619a40386bca661a31b3859fb01ee1c4/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Genentech/assaybench/resolve/988daee0619a40386bca661a31b3859fb01ee1c4/assaybench.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Genentech/assaybench/Genentech/assaybench.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Genentech/assaybench/resolve/988daee0619a40386bca661a31b3859fb01ee1c4/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=Genentech/assaybench "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/data

Generating train split:   0%|          | 0/19 [00:00<?, ? examples/s]

Number of screens in train: 1349
Number of screens in val: 218
Number of screens in test: 334
Number of screens in novel dataset: 19


# Scoring model

Here we provide an example of how to load a model and compute AnDCG@100 on the test set. We use a simple baseline that always predicts the 100 most common genes in the training set.

In [4]:
from assaybench.benchmark.metrics import RankingMetrics

In [5]:
# Here we define out model: a function that take in a prompt and outputs a list of genes)
top_list = [ex["answer"].split(",") for ex in train]
top_list = [item.strip() for sublist in top_list for item in sublist]
# select top 100 most common answers
from collections import Counter
counter = Counter(top_list)
most_common = counter.most_common(100)
most_common_answers = [x[0] for x in most_common]
print(f"Most common answers: {most_common_answers}")

# our model
def top_100_answers(prompt):
    return most_common_answers

Most common answers: ['MYC', 'AURKB', 'UBL5', 'NF2', 'PRAMEF6', 'RPL15', 'RPL18A', 'PLK1', 'LCE1E', 'RAN', 'RPS28', 'RPL32', 'SF3B5', 'CDK1', 'RPL13', 'RPL34', 'RPS19', 'RPS12', 'RPS15', 'RPS15A', 'RPL31', 'RPL7', 'PDCD10', 'RPL5', 'SNRPE', 'RGPD6', 'RPL8', 'ATP6V0C', 'PSMA6', 'PRPF38A', 'RPL37A', 'CHEK1', 'SNRPG', 'NPIPA5', 'RPL21', 'UBE2M', 'RPS20', 'RPS18', 'PRPF19', 'PTPMT1', 'SARS1', 'RPS11', 'BCL2L2-PABPN1', 'POLR2A', 'PRELID1', 'TAOK1', 'PLGLB2', 'RPL23A', 'RPL3', 'SMU1', 'DTL', 'RPL17', 'CDK7', 'RPL12', 'CDC27', 'USP17L5', 'XPO1', 'RPS8', 'RPS14', 'SNU13', 'POLR2K', 'RSL24D1', 'RPS3', 'KIF11', 'FAM86B1', 'YARS1', 'PSMA7', 'RPL26', 'ATR', 'FRYL', 'SF3B3', 'RPP21', 'DDX56', 'KEAP1', 'ATP6V1A', 'FAU', 'RPS29', 'RPL27A', 'SNRPD1', 'MTOR', 'PHB2', 'SNRPD3', 'PKMYT1', 'TRRAP', 'PRAMEF5', 'CAB39', 'RPL36', 'RPS3A', 'UBE2I', 'RPL19', 'SF1', 'BCL2L1', 'RPL11', 'PSMA3', 'H2BC7', 'TUBB', 'DNM2', 'RRM1', 'SNRPF', 'ATP6V1B2']


We evaluate all metrics at k=10 and k=100 and extract the AnDCG@100 metric on the validation set.

In [6]:
metric_fn = RankingMetrics(k_values=[10,100])

metrics = {ex["dataset_name"]: metric_fn.evaluate(top_100_answers(ex["question"]),
                                                  ex["relevance_genes"],
                                                  ex["relevance_scores"]) for ex in val}

adncg_at_100 = [m["adjusted_ndcg@100"] for m in metrics.values()]
print(f"Average adjusted nDCG@100: {sum(adncg_at_100)/len(adncg_at_100)}")

Average adjusted nDCG@100: 0.10596332571004491
